In [36]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score
import joblib

In [37]:
df=pd.read_csv('../data/bank_customer_data.csv')

In [38]:
df.head()

,customer_id,age,gender,marital_status,education,employment_type,years_employed,annual_income,monthly_expenses,num_dependents,...,existing_loans,missed_payments,savings_balance,investment_flag,loan_amount,loan_tenure,interest_rate,default,churn,credit_risk
0,CUST00001,56,Female,Married,Graduate,Salaried,33,627221,27346,0,...,1,1,292746,1,708007.0,21,18.45,0,0,Medium
1,CUST00002,69,Male,Single,Graduate,Salaried,40,668372,34942,2,...,2,0,137967,0,788237.0,21,15.79,0,0,Medium
2,CUST00003,46,Male,Single,High School,Self-Employed,26,554249,26827,3,...,1,1,60548,0,1060949.0,29,22.06,1,1,High
3,CUST00004,32,Female,Single,Graduate,Self-Employed,7,593807,34816,0,...,5,1,289507,0,1103022.0,4,20.94,0,1,Medium
4,CUST00005,60,Male,Married,High School,Salaried,31,396889,13767,5,...,0,0,155872,1,938617.0,7,19.68,0,0,Medium


In [39]:
print(f" the shape of the dataset is {df.shape}")

 the shape of the dataset is (50000, 21)


In [1]:
# dropping the useless features
df.drop(['customer_id','gender','marital_status','num_dependents','loan_tenure'],axis=1,inplace=True)

NameError: name 'df' is not defined

In [41]:
# encode categoricals which are employment type and education
df['education']=df['education'].map({
    'Graduate':1,
    'High School':0,
    'Postgraduate':2
})

In [42]:
df=pd.get_dummies(df,columns=['employment_type'],drop_first=True)

In [43]:
X=df.drop(['default','credit_risk','loan_amount','churn'],axis=1)
y=df['churn']

In [45]:
print(X.shape)
print(X.columns)

(50000, 13)
Index(['age', 'education', 'years_employed', 'annual_income',
       'monthly_expenses', 'credit_score', 'existing_loans', 'missed_payments',
       'savings_balance', 'investment_flag', 'interest_rate',
       'employment_type_Self-Employed', 'employment_type_Unemployed'],
      dtype='str')


In [47]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,stratify=y,test_size=0.3)

In [50]:
models={
    'LogisticRegression':Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=1000))]),
    'SVM':Pipeline([('scaler',StandardScaler()),('model',SVC())]),
    'KNN':Pipeline([('scaler',StandardScaler()),('model',KNeighborsClassifier())]),
    'Decision Tree':DecisionTreeClassifier(),
    'RandomForest':RandomForestClassifier(),
    'XGBOOST':XGBClassifier(eval_metric='logloss')
}
for name,model in models.items():
    model.fit(X_train,y_train)
    y_train_pred=model.predict(X_train)
    y_pred=model.predict(X_test)
    print(f" the training accuracy of the model {name} is {accuracy_score(y_train_pred,y_train)} and testing accuracy of {name} model is {accuracy_score(y_test,y_pred)} ")

 the training accuracy of the model LogisticRegression is 0.7397714285714285 and testing accuracy of LogisticRegression model is 0.7378666666666667 
 the training accuracy of the model SVM is 0.7448 and testing accuracy of SVM model is 0.7386666666666667 
 the training accuracy of the model KNN is 0.7927142857142857 and testing accuracy of KNN model is 0.7036 
 the training accuracy of the model Decision Tree is 1.0 and testing accuracy of Decision Tree model is 0.6488 
 the training accuracy of the model RandomForest is 1.0 and testing accuracy of RandomForest model is 0.7326 
 the training accuracy of the model XGBOOST is 0.8218857142857143 and testing accuracy of XGBOOST model is 0.7292 


In [51]:
for name in ['LogisticRegression','SVM']:
    y_pred=models[name].predict(X_test)
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred))


LogisticRegression:
              precision    recall  f1-score   support

           0       0.75      0.95      0.84     11022
           1       0.52      0.14      0.22      3978

    accuracy                           0.74     15000
   macro avg       0.64      0.55      0.53     15000
weighted avg       0.69      0.74      0.68     15000


SVM:
              precision    recall  f1-score   support

           0       0.75      0.96      0.84     11022
           1       0.53      0.12      0.20      3978

    accuracy                           0.74     15000
   macro avg       0.64      0.54      0.52     15000
weighted avg       0.69      0.74      0.67     15000



In [64]:
imb_ratio=df['churn'].value_counts()[0]/df['churn'].value_counts()[1]
models_balanced={
    'LogisticRegression': Pipeline([
        ('scaler',StandardScaler()),
        ('model',LogisticRegression(class_weight='balanced',max_iter=1000))
    ]),
    'SVM':Pipeline([
        ('scaler',StandardScaler()),
        ('model',SVC(class_weight='balanced'))
    ]),
    'XGBOOST':Pipeline([
        ('scaler',StandardScaler()),
        ('model',XGBClassifier(eval_metric='logloss',scale_pos_weight=round(imb_ratio)))
    ])
}
for name ,model in models_balanced.items():
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    print(f"\n{name}:")
    print(classification_report(y_test,y_pred))


LogisticRegression:
              precision    recall  f1-score   support

           0       0.85      0.62      0.72     11022
           1       0.40      0.70      0.51      3978

    accuracy                           0.65     15000
   macro avg       0.63      0.66      0.62     15000
weighted avg       0.73      0.65      0.67     15000


SVM:
              precision    recall  f1-score   support

           0       0.86      0.58      0.69     11022
           1       0.39      0.75      0.51      3978

    accuracy                           0.62     15000
   macro avg       0.63      0.66      0.60     15000
weighted avg       0.74      0.62      0.64     15000


XGBOOST:
              precision    recall  f1-score   support

           0       0.83      0.64      0.72     11022
           1       0.39      0.64      0.48      3978

    accuracy                           0.64     15000
   macro avg       0.61      0.64      0.60     15000
weighted avg       0.71      0.64    

In [81]:
params_rf={
    'n_estimators':[10,50,75,100],
    'max_depth':[3,5,7]
}
grid_model_rf=GridSearchCV(estimator=RandomForestClassifier(class_weight='balanced'),param_grid=params_rf,scoring='f1_weighted',cv=5,n_jobs=-1)
grid_model_rf.fit(X_train,y_train)
print(grid_model_rf.best_params_)
print(grid_model_rf.best_score_)
print(grid_model_rf.best_estimator_)

{'max_depth': 3, 'n_estimators': 100}
0.6637040870003322
RandomForestClassifier(class_weight='balanced', max_depth=3)


In [82]:
params_xg={
    'n_estimators':[10,50,75,100],
    'max_depth':[3,5,7]
}
grid_model_xgb=GridSearchCV(estimator=XGBClassifier(scale_pos_weight=round(imb_ratio)),param_grid=params_xg,scoring='f1_weighted',cv=5,n_jobs=-1)
grid_model_xgb.fit(X_train,y_train)
print(grid_model_xgb.best_params_)
print(grid_model_xgb.best_score_)
print(grid_model_xgb.best_estimator_)

{'max_depth': 7, 'n_estimators': 100}
0.6677387848004521
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)


In [86]:

for name , model in [('RandomForestClassifier',grid_model_rf),('XGBClassifier',grid_model_xgb)]:
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    print(f"\n{name}")
    print(classification_report(y_test,y_pred))


RandomForestClassifier
              precision    recall  f1-score   support

           0       0.85      0.63      0.72     11022
           1       0.40      0.70      0.51      3978

    accuracy                           0.65     15000
   macro avg       0.63      0.66      0.62     15000
weighted avg       0.73      0.65      0.67     15000


XGBClassifier
              precision    recall  f1-score   support

           0       0.82      0.67      0.74     11022
           1       0.39      0.59      0.47      3978

    accuracy                           0.65     15000
   macro avg       0.60      0.63      0.60     15000
weighted avg       0.70      0.65      0.67     15000



In [87]:
best_model=models_balanced['LogisticRegression']
joblib.dump(best_model,'../models/churn_model.pkl')
print("churn model saved")

churn model saved


In [89]:
coef=best_model.named_steps['model'].coef_[0]
print(coef)

[-0.02800385  0.00759718  0.05385096 -0.11151029  0.05239933 -0.03573709
  0.19166796 -0.00764103  0.00383161 -0.5477828   0.30855799  0.01661194
  0.15890832]


In [92]:
feature_imp=pd.Series(np.abs(coef),index=X.columns).sort_values(ascending=False)
print(f"\n top features for churn prediction")
print(feature_imp)


 top features for churn prediction
investment_flag                  0.547783
interest_rate                    0.308558
existing_loans                   0.191668
employment_type_Unemployed       0.158908
annual_income                    0.111510
years_employed                   0.053851
monthly_expenses                 0.052399
credit_score                     0.035737
age                              0.028004
employment_type_Self-Employed    0.016612
missed_payments                  0.007641
education                        0.007597
savings_balance                  0.003832
dtype: float64
